# PhishGuard - EDA, model comparison and robustness

This notebook reads the artefacts produced by the pipeline (`python ml/eda.py` and `python ml/train.py`) so the
analysis can be reproduced and extended without re-training. Run it from the repository root or from `ml/notebooks`.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
while not (ROOT / "ml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
REPORTS = ROOT / "ml" / "reports"
FIG = REPORTS / "figures"
eda = json.loads((REPORTS / "eda.json").read_text())
res = json.loads((REPORTS / "results.json").read_text())
print("model version:", res["model_version"])

## 1. Datasets and class balance

In [ ]:
pd.DataFrame({k: {"rows": v["rows"], "features": v["n_features"], **v["class_balance"], "duplicate rows": v["duplicate_feature_rows"]}
              for k, v in eda["datasets"].items()}).T

In [ ]:
display(Image(FIG / "class_balance.png"))

## 2. Dataset artefacts and leakage

PhiUSIIL's legitimate URLs are all bare `https://www.<domain>` homepages, and `URLSimilarityIndex` nearly separates
the classes on its own. Both would inflate scores without generalising to real links.

In [ ]:
pd.DataFrame(eda["artefacts"]["phiusiil_url_artefacts"]).drop(index=[], errors="ignore")

In [ ]:
pd.DataFrame(eda["artefacts"]["phiusiil_single_feature_auc"]).head(15)

In [ ]:
display(Image(FIG / "leakage_single_feature_auc.png"))

## 3. Correlations and distributions

In [ ]:
for k in ("uci", "url", "content"):
    print(k); display(pd.DataFrame(eda["datasets"][k]["label_correlation"]).head(10))

In [ ]:
display(Image(FIG / "dist_url.png")); display(Image(FIG / "corr_url.png"))

## 4. Model comparison (held-out test set)

In [ ]:
def table(key):
    fs = res["feature_sets"][key]
    rows = [{"model": m["name"], "cv_f1": m["cv"]["f1"]["mean"], **{k: v for k, v in m["test"].items() if k != "confusion_matrix"},
             "train_s": m["train_time_s"], "infer_ms_per_1k": m["inference_ms_per_1k"]} for m in fs["models"]]
    print(fs["title"], "| deployed:", fs["best"])
    return pd.DataFrame(rows).sort_values("f1", ascending=False).round(4)

for k in ("url", "content", "uci"):
    if k in res["feature_sets"]:
        display(table(k))

In [ ]:
for k in ("url", "content", "uci"):
    display(Image(FIG / f"roc_pr_{k}.png")); display(Image(FIG / f"confusion_{k}.png"))

## 5. Explainability (SHAP) of the deployed models

In [ ]:
for k in ("url", "content"):
    print(k, res["feature_sets"][k]["best"])
    display(pd.DataFrame(res["feature_sets"][k]["shap"]["mean_abs"]).head(12))
    display(Image(FIG / f"shap_beeswarm_{k}.png"))

## 6. Robustness checks

In [ ]:
for k, fs in res["feature_sets"].items():
    r = fs["robustness"]
    print("==", k)
    if "split_comparison" in r:
        print(" random split F1:", round(r["split_comparison"]["random_split"]["f1"], 4),
              "| domain-grouped F1:", round(r["split_comparison"]["domain_grouped_split"]["f1"], 4))
    if "duplicate_leakage" in r:
        d = r["duplicate_leakage"]
        print(" test rows also in train:", d["test_rows_also_in_train"], f"({d['test_rows_also_in_train_pct']:.1%})",
              "| F1 unseen-only:", round(d["metrics_unseen_test_only"]["f1"], 4),
              "| dedup benchmark F1:", round(r["deduplicated_benchmark"]["metrics"]["f1"], 4))
    if "cross_dataset" in r:
        for name, m in r["cross_dataset"].items():
            print(" ", name, "ROC-AUC", round(m["roc_auc"], 4))
    if "ablation_without_size_features" in r:
        print(" without size features F1:", round(r["ablation_without_size_features"]["metrics"]["f1"], 4))
    print(" Brier:", round(r["calibration"]["brier"], 4))

In [ ]:
pd.DataFrame(res["feature_sets"]["url"]["robustness"]["sanity_suite"]["items"])